# MapBiomas Cog — All Initiatives (version_02)

Notebook fino: autentica no Colab e abre a UI do pacote `mapbiomas_cog`.
O download usa o engine **`ee_batch_tiled`** (export por tiles, com manifest e
retry). A mesma autenticação de sempre.

1. Clone + instalação do pacote
2. Autenticação (GCP + Earth Engine)
3. Configuração (`COUNTRIES`/`THEMES`; `[]` = todos do `config.OBJ`)
4. Abrir a interface

In [ ]:
#@title 1. Clone + install
import os, sys, subprocess
REPO = "/content/brazil-fire"
PKG  = f"{REPO}/mapbiomas_fire_monitor/version_02"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--branch", "main",
                    "https://github.com/mapbiomas/brazil-fire.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"])
subprocess.run([sys.executable, "-m", "pip", "install", "-e", PKG])  # sem -q: erros visiveis
sys.path.insert(0, f"{PKG}/src")  # fallback se o editable install falhar
import mapbiomas_cog
print('mapbiomas_cog', mapbiomas_cog.__version__, 'OK')

# deps defensivas (instala so se faltar)
for mod, pkg in (("ee", "earthengine-api"), ("gcsfs", "gcsfs"), ("rasterio", "rasterio")):
    try:
        __import__(mod)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.run([sys.executable, "-m", "pip", "install", pkg])

In [ ]:
#@title 2. Auth (GCP + Earth Engine)
from google.colab import auth
auth.authenticate_user()

import ee
ee.Authenticate()
ee.Initialize(project='mapbiomas-fire-485203')

In [ ]:
#@title 3. Configuration
# Country tabs: [] = discover ALL from config.OBJ; or list explicit OBJ codes.
# Available country codes: brasil, argentina, bolivia, chile, colombia, ecuador,
#   indonesia, paraguay, peru, venezuela  (e.g. ["brasil", "ecuador"]).
COUNTRIES = []

# Themes: [] = ALL themes; or e.g. ["fire"].
# Available themes: fire, lulc, lulc_10m, soil, water, atmosphere, climate_risk, urban
THEMES = []

# Runner: 'ee_batch_tiled' (default). 'hv_local'/'dataflow' virão depois.
RUNNER = 'ee_batch_tiled'

import sys; sys.path.append('/content/brazil-fire/mapbiomas_fire_monitor/version_02/src')
from mapbiomas_cog.domain import config as cog_config
obj = cog_config.load_obj()
print('Paises:', cog_config.resolve_countries(obj, COUNTRIES, THEMES))
print('Runner:', RUNNER)

In [ ]:
#@title 4. Interface
# Navegue: pais -> tema -> colecao -> produto.
# Load units -> Export (submete tiles) -> Sync (poll) -> Assemble (COG).
from mapbiomas_cog.ui import run_ui

ui = run_ui(COUNTRIES, THEMES, runner=RUNNER)